In [1]:
import xarray as xr
from pathlib import Path
from xhistogram.xarray import histogram as xhist
import numpy as np

In [2]:
from dask.distributed import Client
client = Client(n_workers=2, threads_per_worker=3, memory_limit=15e9)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 6,Total memory: 27.94 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42513,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41539,Total threads: 3
Dashboard: http://127.0.0.1:34333/status,Memory: 13.97 GiB
Nanny: tcp://127.0.0.1:33661,


In [3]:
stores1 = sorted(Path("/work/bk1450/b383184/Amazon/Atlantic/data/tracks_4567").glob("Parcels_run_*_*.zarr"))
stores2  = sorted(Path("/work/bk1450/b383184/Amazon/Atlantic/data/tracks_3456").glob("Parcels_run_*_*.zarr"))
stores3 = sorted(Path("/work/bk1450/b383184/Amazon/Atlantic/data/tracks_2345").glob("Parcels_run_*_*.zarr"))

stores = stores1+stores2+stores3
stores[:3]

[PosixPath('/work/bk1450/b383184/Amazon/Atlantic/data/tracks_4567/Parcels_run_4567_2022-06-05.zarr'),
 PosixPath('/work/bk1450/b383184/Amazon/Atlantic/data/tracks_4567/Parcels_run_4567_2022-06-10.zarr'),
 PosixPath('/work/bk1450/b383184/Amazon/Atlantic/data/tracks_4567/Parcels_run_4567_2022-06-15.zarr')]

In [11]:
ds_list = [xr.open_zarr(s) for s in stores]
ds = xr.concat(ds_list,dim='trajectory')

ds = ds.drop_vars(['z','age','sal','temp'])
ds

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/dask/array/core.py:4998: PerformanceWarning: Increasing number of chunks by factor of 46
  result = blockwise(
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/dask/array/core.py:4998: PerformanceWarning: Increasing number of chunks by factor of 46
  result = blockwise(
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/dask/array/core.py:4998: PerformanceWarning: Increasing number of chunks by factor of 46
  result = blockwise(


<xarray.Dataset> Size: 91GB
Dimensions:     (trajectory: 6160000, obs: 925)
Coordinates:
  * obs         (obs) int32 4kB 0 1 2 3 4 5 6 7 ... 918 919 920 921 922 923 924
  * trajectory  (trajectory) int64 49MB 0 1 2 3 4 5 ... 9995 9996 9997 9998 9999
Data variables:
    lat         (trajectory, obs) float32 23GB dask.array<chunksize=(10000, 20), meta=np.ndarray>
    lon         (trajectory, obs) float32 23GB dask.array<chunksize=(10000, 20), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 46GB dask.array<chunksize=(10000, 20), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        JITParticleAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [12]:
num_valid_obs_steps = int(ds.lat.notnull().any('trajectory').sum().compute().data[()])
ds = ds.isel(obs=slice(None, num_valid_obs_steps))
ds

<xarray.Dataset> Size: 73GB
Dimensions:     (trajectory: 6160000, obs: 741)
Coordinates:
  * obs         (obs) int32 3kB 0 1 2 3 4 5 6 7 ... 734 735 736 737 738 739 740
  * trajectory  (trajectory) int64 49MB 0 1 2 3 4 5 ... 9995 9996 9997 9998 9999
Data variables:
    lat         (trajectory, obs) float32 18GB dask.array<chunksize=(10000, 20), meta=np.ndarray>
    lon         (trajectory, obs) float32 18GB dask.array<chunksize=(10000, 20), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 37GB dask.array<chunksize=(10000, 20), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        JITParticleAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [13]:
import numpy as np
import pandas as pd

In [14]:
def snap_to_5day_grid(dates):
    """
    Snap dates to nearest 5-day grid: 5th, 10th, 15th, 20th, 25th, 30th of each month.
    If day is 1st, snap to 30th of previous month (or last valid day).
    """
    dates = pd.to_datetime(dates)
    result = []
    
    for date in dates:
        day = date.day
        year = date.year
        month = date.month
        
        # Target days in each month
        targets = [5, 10, 15, 20, 25, 30]
        
        # Special case: if day is 1, snap to 30th of previous month
        if day == 1:
            if month == 1:
                # Go to December of previous year
                prev_year, prev_month = year - 1, 12
            else:
                prev_year, prev_month = year, month - 1
            
            # Try 30th, then 25th if February
            for target_day in [30, 25]:
                try:
                    result.append(pd.Timestamp(prev_year, prev_month, target_day))
                    break
                except ValueError:
                    continue
            continue
        
        # Find closest target day
        distances = [abs(day - target) for target in targets]
        closest_idx = distances.index(min(distances))
        closest_day = targets[closest_idx]
        
        # Check if the closest day exists in this month
        try:
            snapped = pd.Timestamp(year, month, closest_day)
            result.append(snapped)
        except ValueError:
            # Day doesn't exist (e.g., Feb 30), use the last valid target
            for target in reversed(targets):
                try:
                    snapped = pd.Timestamp(year, month, target)
                    result.append(snapped)
                    break
                except ValueError:
                    continue
    
    return np.array(result, dtype='datetime64[ns]')

In [29]:
t = pd.to_datetime(ds.start_time).floor('D')  # DatetimeIndex

# 1st-of-month -> previous 30th
is_first = (t.day == 1)
t = t.where(~is_first, t - pd.offsets.MonthBegin(1) - pd.Timedelta(days=1))

base = pd.Timestamp('1970-01-01')
days = (t - base).days
rounded = base + pd.to_timedelta(((days + 2)//5)*5, unit='D')

# enforce 5/10/15/20/25/30
valid = rounded.day.isin([5,10,15,20,25,30])
rounded = rounded.where(valid, rounded - pd.Timedelta(days=5))

ds['start_time_round5'] = rounded

rounded
# ds['start_time_round5'] = rounded

DatetimeIndex(['2022-06-02', '2022-06-02', '2022-06-02', '2022-06-02',
               '2022-06-02', '2022-06-02', '2022-06-02', '2022-06-02',
               '2022-06-02', '2022-06-02',
               ...
               '2025-06-01', '2025-06-01', '2025-06-01', '2025-06-01',
               '2025-06-01', '2025-06-01', '2025-06-01', '2025-06-01',
               '2025-06-01', '2025-06-01'],
              dtype='datetime64[ns]', length=6160000, freq=None)

In [15]:
ds = ds.assign(start_time=ds.time.isel(obs=0).compute())

time5days = snap_to_5day_grid(ds.start_time.values)
ds['start_time_round5'] = time5days
ds

<xarray.Dataset> Size: 73GB
Dimensions:            (trajectory: 6160000, obs: 741,
                        start_time_round5: 6160000)
Coordinates:
  * obs                (obs) int32 3kB 0 1 2 3 4 5 6 ... 735 736 737 738 739 740
  * trajectory         (trajectory) int64 49MB 0 1 2 3 4 ... 9996 9997 9998 9999
  * start_time_round5  (start_time_round5) datetime64[ns] 49MB 2022-06-05 ......
Data variables:
    lat                (trajectory, obs) float32 18GB dask.array<chunksize=(10000, 20), meta=np.ndarray>
    lon                (trajectory, obs) float32 18GB dask.array<chunksize=(10000, 20), meta=np.ndarray>
    time               (trajectory, obs) datetime64[ns] 37GB dask.array<chunksize=(10000, 20), meta=np.ndarray>
    start_time         (trajectory) datetime64[ns] 49MB 2022-06-05 ... 2025-0...
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        JITParticleAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [ ]:
ds = ds.assign(start_year=ds.start_time_round5.dt.year)
ds = ds.assign(start_month=ds.start_time_round5.dt.month)
ds = ds.assign(start_day=ds.start_time_round5.dt.day)

In [35]:
np.unique(ds.start_time)

array(['2022-06-05T00:00:00.000000000', '2022-06-10T00:00:00.000000000',
       '2022-06-15T00:00:00.000000000', '2022-06-20T00:00:00.000000000',
       '2022-06-25T00:00:00.000000000', '2022-06-30T00:00:00.000000000',
       '2022-07-05T00:00:00.000000000', '2022-07-10T00:00:00.000000000',
       '2022-07-15T00:00:00.000000000', '2022-07-20T00:00:00.000000000',
       '2022-07-25T00:00:00.000000000', '2022-07-30T00:00:00.000000000',
       '2022-08-04T00:00:00.000000000', '2022-08-09T00:00:00.000000000',
       '2022-08-14T00:00:00.000000000', '2022-08-19T00:00:00.000000000',
       '2022-08-24T00:00:00.000000000', '2022-08-29T00:00:00.000000000',
       '2022-09-03T00:00:00.000000000', '2022-09-08T00:00:00.000000000',
       '2022-09-13T00:00:00.000000000', '2022-09-18T00:00:00.000000000',
       '2022-09-23T00:00:00.000000000', '2022-09-28T00:00:00.000000000',
       '2022-10-03T00:00:00.000000000', '2022-10-08T00:00:00.000000000',
       '2022-10-13T00:00:00.000000000', '2022-10-18

In [37]:
print(np.unique(ds.start_time)[205:210])

print('round5')
print(np.unique(ds.start_time_round5)[205:210])

['2025-02-10T00:00:00.000000000' '2025-02-14T00:00:00.000000000'
 '2025-02-15T00:00:00.000000000' '2025-02-19T00:00:00.000000000'
 '2025-02-20T00:00:00.000000000']
round5
['2025-05-12T00:00:00.000000000' '2025-05-17T00:00:00.000000000'
 '2025-05-22T00:00:00.000000000' '2025-05-27T00:00:00.000000000'
 '2025-06-01T00:00:00.000000000']


In [ ]:
# # Test with a few examples first
# test_dates = [
#     '2022-08-04T00:00:00.000000000',  # Should snap to 2022-08-05
#     '2022-11-02T00:00:00.000000000',  # Should snap to 2022-11-05  
#     '2024-11-01T00:00:00.000000000',  # Should snap to 2024-10-30
#     '2024-12-01T00:00:00.000000000',  # Should snap to 2024-11-30
#     '2023-01-01T00:00:00.000000000',  # Should snap to 2022-12-30
#     '2022-12-18T00:00:00.000000000',  # Should snap to 2022-12-20
#     '2023-02-28T00:00:00.000000000',  # Should snap to 2023-02-25 (Feb has no 30th)
# ]
# snapped_dates = snap_to_5day_grid(test_dates)
# snapped_dates

In [ ]:
lat_min=ds.lat.min().compute().data[()]
lon_min=ds.lon.min().compute().data[()]

lat_max=ds.lat.max().compute().data[()]
lon_max=ds.lon.max().compute().data[()]